In [2]:
"""
CORRECTED Hallucination Prevention Rail - Multiple LLM Responses Approach
===============================================================================
Implementation based on GitHub reference: 
https://github.com/marvik-ai/llama2-nemo-guardrails/blob/main/Hallucination_RAIL.ipynb

KEY CONCEPT:
- User provides a QUERY
- Generate 2-3 alternative responses from LLM for the SAME query (temperature=1)
- Concatenate alternative responses → "paragraph" (context)
- First/main response → "statement" (hypothesis)
- Check if statement agrees with paragraph using self_check_hallucination

WORKFLOW:
Query → LLM (temp=1, n=3) → [Response1, Response2, Response3]
- paragraph = Response2 + Response3 (context)
- statement = Response1 (hypothesis)
- Agreement check: Does Response1 agree with Response2+Response3?
"""

import os
from pathlib import Path
from nemoguardrails import RailsConfig, LLMRails
import shutil
from openai import OpenAI

# ============================================================================
# STEP 1: Setup OpenAI Client
# ============================================================================
os.environ["OPENAI_API_KEY"] = "sk-REDACTED-set-your-own-key"
client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])
print("✅ OpenAI client initialized\n")

# ============================================================================
# STEP 2: Create Hallucination Configuration Directory
# ============================================================================

hallucination_dir_v2 = Path("./config_hallucination")

# Clean up if exists
if hallucination_dir_v2.exists():
    shutil.rmtree(hallucination_dir_v2)
    print(f"🗑️  Removed existing {hallucination_dir_v2}")

hallucination_dir_v2.mkdir(exist_ok=True)
print(f"✅ Created directory: {hallucination_dir_v2.absolute()}\n")

# ============================================================================
# STEP 3: Generate config.yml with Output Rails
# ============================================================================

hallucination_config_v2 = """# Hallucination Prevention Rail - Multiple Response Generation
# Based on: https://github.com/marvik-ai/llama2-nemo-guardrails

models:
  - type: main
    engine: openai
    model: gpt-4o-mini

rails:
  output:
    flows:
      - self check hallucination

prompts:
  - task: self_check_hallucination
    content: |-
      You are given a task to identify if the hypothesis is in agreement with the context below.
      You will only use the contents of the context and not rely on external knowledge.
      Your answer MUST be only "YES" or "NO". Here are some examples:
      
      "context": The car has 4 doors. The car has 2 doors.
      "hypothesis": The car has 5 doors.
      "agreement": NO

      "context": Uruguay has 3.3 million habitants. Uruguay has more than 2 million habitants. Uruguay has less than 3.5 million habitants.
      "hypothesis": Uruguay has at least 2.9 million habitants.
      "agreement": YES

      "context": The unemployment rate in March 2023 was 3.5 percent. The unemployment rate in March 2023 was 6.5 percent.
      "hypothesis": The unemployment rate in March 2023 was 5.5 percent.
      "agreement": NO

      "context": {{ paragraph }} 
      "hypothesis": {{ statement }} 
      "agreement":
"""

config_file_v2 = hallucination_dir_v2 / "config.yaml"
with open(config_file_v2, 'w') as f:
    f.write(hallucination_config_v2)

print(f"✅ Created {config_file_v2}")
print("   - Model: gpt-4o-mini")
print("   - Output Rails: self check hallucination")
print("   - Approach: Generate multiple responses per query")
print("   - Temperature: 1.0 for diversity\n")

# ============================================================================
# STEP 4: Helper Function to Generate Multiple Responses
# ============================================================================

def generate_multiple_responses(query: str, n: int = 3, temperature: float = 1.0) -> list:
    """
    Generate multiple responses for the same query using OpenAI API.
    
    Args:
        query: User query to send to LLM
        n: Number of alternative responses to generate (default: 3)
        temperature: Higher = more diverse responses (default: 1.0)
    
    Returns:
        List of response strings
    """
    try:
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[{"role": "user", "content": query}],
            n=n,
            temperature=temperature,
            max_tokens=150
        )
        
        responses = [choice.message.content.strip() for choice in response.choices]
        return responses
    
    except Exception as e:
        print(f"Error generating responses: {e}")
        return []

print("✅ Helper function defined: generate_multiple_responses()")
print("   - Generates N alternative responses for same query")
print("   - Uses temperature=1.0 for diversity\n")

# ============================================================================
# STEP 5: Load Hallucination Rails Configuration
# ============================================================================

print("Loading hallucination configuration v2...")
hallucination_rails_config_v2 = RailsConfig.from_path(str(hallucination_dir_v2))
print(f"✅ Configuration loaded from {hallucination_dir_v2}\n")

print("Initializing Hallucination Rails v2...")
hallucination_rails_v2 = LLMRails(hallucination_rails_config_v2, verbose=False)
print("✅ Hallucination Rails v2 initialized\n")

print("="*80)
print("READY TO TEST HALLUCINATION RAIL V2 (CORRECTED APPROACH)")
print("="*80)
print("✓ Query → Generate 3 responses with temp=1.0")
print("✓ paragraph = Response2 + Response3")
print("✓ statement = Response1")
print("✓ Check: Does Response1 agree with Response2+Response3?")
print("="*80 + "\n")

✅ OpenAI client initialized

🗑️  Removed existing config_hallucination
✅ Created directory: /workspace/project/neo_dialog_rail/config_hallucination

✅ Created config_hallucination/config.yaml
   - Model: gpt-4o-mini
   - Output Rails: self check hallucination
   - Approach: Generate multiple responses per query
   - Temperature: 1.0 for diversity

✅ Helper function defined: generate_multiple_responses()
   - Generates N alternative responses for same query
   - Uses temperature=1.0 for diversity

Loading hallucination configuration v2...
✅ Configuration loaded from config_hallucination

Initializing Hallucination Rails v2...


/workspace/project/neo_dialog_rail/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ Hallucination Rails v2 initialized

READY TO TEST HALLUCINATION RAIL V2 (CORRECTED APPROACH)
✓ Query → Generate 3 responses with temp=1.0
✓ paragraph = Response2 + Response3
✓ statement = Response1
✓ Check: Does Response1 agree with Response2+Response3?



In [ ]:
# ============================================================================
# STEP 6: Test Cases - Query-based Hallucination Detection
# ============================================================================

async def run_hallucination_tests_v2():
    """
    Test hallucination rail with QUERY-BASED context generation.
    
    For each query:
    1. Generate 3 responses with temperature=1.0
    2. paragraph (context) = Response2 + Response3
    3. statement (hypothesis) = Response1
    4. Check if Response1 agrees with Response2+Response3
    """
    print("="*80)
    print("TESTING HALLUCINATION PREVENTION V2 - QUERY-BASED APPROACH")
    print("="*80)
    print("Method: Generate multiple LLM responses per query")
    print("Context: Concatenated alternative responses (paragraph)")
    print("Hypothesis: First response (statement)")
    print("="*80 + "\n")

    # Test queries - covering different scenarios
    test_queries = [
        # --- Questions with known answers (should have consistent responses) ---
        {
            "query": "What is the daily withdrawal limit at most ATMs?",
            "description": "Factual banking question",
            "expected_agreement": "YES"
        },
        {
            "query": "What is the standard interest rate for savings accounts?",
            "description": "General finance question",
            "expected_agreement": "MAYBE"
        },
        {
            "query": "What are common bank fees?",
            "description": "General knowledge question",
            "expected_agreement": "YES"
        },
        
        # --- Questions without clear answers (should detect hallucination) ---
        {
            "query": "Who is the president of Mars?",
            "description": "Nonsensical question - no president exists",
            "expected_agreement": "NO"
        },
        {
            "query": "What is the exact balance in account number 12345?",
            "description": "Specific info not available - should hallucinate",
            "expected_agreement": "NO"
        },
        {
            "query": "When was the last transaction made on my account?",
            "description": "User-specific data not available",
            "expected_agreement": "NO"
        },
        
        # --- Ambiguous or variable answers ---
        {
            "query": "What is a good investment strategy for retirement?",
            "description": "Open-ended question with many valid answers",
            "expected_agreement": "MAYBE"
        },
        {
            "query": "Should I invest in cryptocurrency?",
            "description": "Opinion-based question",
            "expected_agreement": "MAYBE"
        },
        
        # --- Factual finance questions (consistent answers expected) ---
        {
            "query": "What does ROI stand for?",
            "description": "Finance acronym - should be consistent",
            "expected_agreement": "YES"
        },
        {
            "query": "What is FDIC insurance?",
            "description": "Standard finance term",
            "expected_agreement": "YES"
        },
        {
            "query": "How does inflation affect savings?",
            "description": "Finance concept explanation",
            "expected_agreement": "YES"
        },
        
        # --- Made-up or false premise questions ---
        {
            "query": "What is the color of the CEO of Bank of America's car?",
            "description": "Unknowable specific detail",
            "expected_agreement": "NO"
        },
        {
            "query": "How many employees work at branch #99999?",
            "description": "Specific data not available",
            "expected_agreement": "NO"
        },
        {
            "query": "What time does the Mars branch open?",
            "description": "Nonsensical location",
            "expected_agreement": "NO"
        },
        
        # --- Policy/procedure questions (should be consistent) ---
        {
            "query": "How do I open a savings account?",
            "description": "Standard banking procedure",
            "expected_agreement": "YES"
        }
    ]

    results = {}

    for i, test_case in enumerate(test_queries, 1):
        query = test_case["query"]
        description = test_case["description"]
        expected = test_case["expected_agreement"]
        
        print(f"Test {i}/{len(test_queries)}: {description}")
        print("-" * 80)
        print(f"Query: {query}")
        print(f"Expected Agreement: {expected}")
        print()
        
        # STEP 1: Generate multiple responses for the query
        print("  Generating 3 alternative responses (temp=1.0)...")
        alternative_responses = generate_multiple_responses(query, n=3, temperature=1.0)
        
        if len(alternative_responses) < 3:
            print(f"  ⚠️  Only got {len(alternative_responses)} responses, skipping...")
            continue
        
        # STEP 2: Create paragraph (context) and statement (hypothesis)
        statement = alternative_responses[0]  # First response = hypothesis
        paragraph = " ".join(alternative_responses[1:])  # Remaining responses = context
        
        print(f"  Statement (Response 1): {statement}...")
        print(f"  Paragraph (Response 2+3): {paragraph}...")
        print()
        
        # STEP 3: Use hallucination rail to check agreement
        # We need to manually invoke the self_check_hallucination task
        # because we're providing pre-generated responses
        
        check_prompt = hallucination_rails_v2.runtime.llm_task_manager.render_task_prompt(
            task="self_check_hallucination",
            context={
                "statement": statement,
                "paragraph": paragraph
            }
        )
        
        # Get agreement result
        from nemoguardrails.actions.llm.utils import llm_call
        agreement_result = await llm_call(hallucination_rails_v2.llm, check_prompt)
        agreement_result = agreement_result.strip().upper()
        
        print(f"  Agreement Result: {agreement_result}")
        print()
        
        # Store result
        results[f"test_{i}"] = {
            "query": query,
            "description": description,
            "statement": statement,
            "paragraph": paragraph,
            "agreement": agreement_result,
            "expected": expected
        }

    print("="*80)
    print(f"RESULTS: {len(test_queries)} tests completed")
    print("="*80)
    
    return results

# ============================================================================
# STEP 7: Execute Test Suite
# ============================================================================

print("\n🚀 Running Query-Based Hallucination Tests...\n")
hallucination_results_v2 = await run_hallucination_tests_v2()

# ============================================================================
# STEP 8: Detailed Summary
# ============================================================================
print("\n" + "="*80)
print("🎉 HALLUCINATION RAIL V2 TEST SUMMARY 🎉")
print("="*80)
print(f"\nConfiguration Directory: {hallucination_dir_v2.absolute()}") 
print(f"\nApproach:")
print(f"  ✓ Query → Generate 3 responses (temp=1.0)")
print(f"  ✓ paragraph = Response2 + Response3 (context)")
print(f"  ✓ statement = Response1 (hypothesis)")
print(f"  ✓ Agreement Check: YES/NO using self_check_hallucination")

print(f"\nTest Results:")
total_v2 = len(hallucination_results_v2)

print(f"  Total Tests: {total_v2}")

print("\nDetailed Results by Category:")

# Group by expected agreement
yes_tests = [r for r in hallucination_results_v2.values() if r["expected"] == "YES"]
no_tests = [r for r in hallucination_results_v2.values() if r["expected"] == "NO"]
maybe_tests = [r for r in hallucination_results_v2.values() if r["expected"] == "MAYBE"]

print(f"\n  Expected YES (consistent answers): {len(yes_tests)} tests")
print(f"  Expected NO (hallucination detection): {len(no_tests)} tests")
print(f"  Expected MAYBE (variable answers): {len(maybe_tests)} tests")

print(f"\n✅ All {total_v2} tests completed!")


🚀 Running Query-Based Hallucination Tests...

TESTING HALLUCINATION PREVENTION V2 - QUERY-BASED APPROACH
Method: Generate multiple LLM responses per query
Context: Concatenated alternative responses (paragraph)
Hypothesis: First response (statement)

Test 1/15: Factual banking question
--------------------------------------------------------------------------------
Query: What is the daily withdrawal limit at most ATMs?
Expected Agreement: YES

  Generating 3 alternative responses (temp=1.0)...
  Statement (Response 1): The daily withdrawal limit at ATMs can vary widely based on several factors, including the bank or financial institution, the type of account, and the specific ATM. Generally, the common range for daily withdrawal limits is between $300 and $1,000. 

Some banks may offer higher limits for certain account types or customers, while others may have lower limits for security reasons. If you need to know the specific limit for your account, it is best to check with your ban

NameError: name 'test_queries' is not defined

In [ ]:
# ============================================================================
# Try Your Own Query
# ============================================================================

async def test_single_query(query: str):
    """
    Test hallucination detection for a single query.
    Demonstrates the complete workflow clearly.
    """
    print("="*80)
    print("HALLUCINATION DETECTION DEMO")
    print("="*80)
    print(f"\n📝 Query: {query}\n")
    
    # Generate 3 alternative responses
    print("🔄 Generating 3 alternative responses (temperature=1.0)...\n")
    responses = generate_multiple_responses(query, n=3, temperature=1.0)
    
    if len(responses) < 3:
        print("❌ Failed to generate enough responses")
        return
    
    # Display all responses
    for i, resp in enumerate(responses, 1):
        print(f"Response {i}:")
        print(f"  {resp}")
        print()
    
    # Create statement and paragraph
    statement = "The color of the CEO of Bank of America's car is red"
    paragraph = " ".join(responses[0:])
    
    print("-" * 80)
    print("\n🎯 Hallucination Check Setup:")
    print(f"\nStatement (Hypothesis): {statement}")
    print(f"\nParagraph (Context): {paragraph}")
    print()
    
    # Check for hallucination
    print("🔍 Checking if statement agrees with paragraph...\n")
    
    check_prompt = hallucination_rails_v2.runtime.llm_task_manager.render_task_prompt(
        task="self_check_hallucination",
        context={
            "statement": statement,
            "paragraph": paragraph
        }
    )
    
    from nemoguardrails.actions.llm.utils import llm_call
    agreement = await llm_call(hallucination_rails_v2.llm, check_prompt)
    agreement = agreement.strip().upper()
    
    print(f"Agreement Result: {agreement}")
    print()
    
    if "YES" in agreement:
        print("✅ CONSISTENT: The responses agree with each other")
        print("   → Low risk of hallucination")
    elif "NO" in agreement:
        print("⚠️  INCONSISTENT: The responses contradict each other")
        print("   → High risk of hallucination - answer may not be reliable")
    else:
        print("❓ UNCERTAIN: Could not determine agreement")
    
    print("\n" + "="*80)

# ============================================================================
# Example: Test with a question that has no real answer
# ============================================================================

print("\n🧪 Example 1: Testing with nonsensical question\n")
await test_single_query("What is the color of the CEO of Bank of America's car?")


🧪 Example 1: Testing with nonsensical question

HALLUCINATION DETECTION DEMO

📝 Query: What is the color of the CEO of Bank of America's car?

🔄 Generating 3 alternative responses (temperature=1.0)...



NameError: name 'generate_multiple_responses' is not defined

In [5]:
from nemoguardrails import RailsConfig, LLMRails
from nemoguardrails.actions.llm.utils import get_multiline_response, llm_call, strip_quotes
from nemoguardrails.llm.taskmanager import LLMTaskManager
from nemoguardrails.llm.types import Task
import os
import logging
# Suppress verbose output
os.environ["NEMOGUARDRAILS_VERBOSE"] = "False"
logging.getLogger("nemoguardrails").setLevel(logging.CRITICAL)
logging.getLogger("httpx").setLevel(logging.CRITICAL)
logging.getLogger("openai").setLevel(logging.CRITICAL)

from typing import Optional
from langchain_core.prompts import PromptTemplate
from langchain_classic.chains import LLMChain
from nemoguardrails.actions.llm.utils import (
    get_multiline_response,
    llm_call,
    strip_quotes,
)
from nemoguardrails.llm.taskmanager import LLMTaskManager
from nemoguardrails.llm.types import Task

log = logging.getLogger(__name__)

HALLUCINATION_NUM_EXTRA_RESPONSES = 2

# ============================================================================
# 1. API Setup
# ============================================================================
os.environ["OPENAI_API_KEY"] = "sk-REDACTED-set-your-own-key"

config_content = """
models:
  - type: main
    engine: openai
    model: gpt-4o-mini

prompts:
  - task: check_hallucination
    content: |-
      You are given a task to identify if the hypothesis is in agreement with the context below.
      You will only use the contents of the context and not rely on external knowledge.
      Your answer MUST be only "YES" or "NO". Here are some examples:

      "context": The car has 4 doors. The car has 2 doors.
      "hypothesis": The car has 5 doors.
      "agreement": NO

      "context": Uruguay has 3.3 million habitants. Uruguay has more than 2 million habitants. Uruguay has less than 3.5 million habitants.
      "hypothesis": Uruguay has at least 2.9 million habitants.
      "agreement": YES

      "context": The unemployment rate in March 2023 was 3.5 percent. The unemployment rate in March 2023 was 6.5 percent.
      "hypothesis": The unemployment rate in March 2023 was 5.5 percent.
      "agreement": NO

      "context": {{ paragraph }} 
      "hypothesis": {{ statement }} 
      "agreement":
"""

colang_content = """
define flow check hallucination
    bot ...
    $result = execute check_hallucination
    if $result
        bot inform answer prone to hallucination

define bot inform answer prone to hallucination
    "The previous answer is prone to hallucination and may not be accurate. Please double check the answer using additional sources."
"""

HALLUCINATION_NUM_EXTRA_RESPONSES = 2


async def check_hallucination(
    llm_task_manager: LLMTaskManager,
    context: Optional[dict] = None,
    llm = None,
    use_llm_checking: bool = True,
):
    """Checks if the last bot response is a hallucination by checking multiple completions for self-consistency.

    :return: True if hallucination is detected, False otherwise.
    """

    bot_response = context.get("last_bot_message")
    last_user_message_string = context.get("last_user_message")
    num_responses = HALLUCINATION_NUM_EXTRA_RESPONSES
    
    # Use the "generate" call from langchain to get all completions in the same response.
    last_bot_prompt = PromptTemplate(template="{text}", input_variables=["text"])
    chain = LLMChain(prompt=last_bot_prompt, llm=llm)
    
    extra_responses = []
    for i in range(num_responses):
        result = chain.run(last_user_message_string)
        print(f"Extra response {i+1}: {result}")
        result = get_multiline_response(result)
        print(f"Processed extra response {i+1}: {result}")
        result = strip_quotes(result)
        print(f"Stripped extra response {i+1}: {result}")
        extra_responses.append(result)
    
    if len(extra_responses) == 0:
        print(f"No extra LLM responses were generated for '{bot_response}' hallucination check.")
        return False
    elif len(extra_responses) < num_responses:
        print(f"Requested {num_responses} extra LLM responses for hallucination check, "
            f"received {len(extra_responses)}.")
        
    if use_llm_checking:
        # Only support LLM-based agreement check in current version
        prompt = llm_task_manager.render_task_prompt(
            task=Task.CHECK_HALLUCINATION,
            context={
                "statement": bot_response,
                "paragraph": " ".join(extra_responses),
            },
        )

        
        agreement = await llm_call(llm, prompt)

        agreement = agreement.lower().strip()
        print(f"Agreement result for looking for hallucination is {agreement}.")
        # Return True if the hallucination check fails
        return "no" in agreement

    return False

# initialize rails config
config = RailsConfig.from_content(colang_content=colang_content, yaml_content=config_content)

# create rails
app = LLMRails(config, verbose=True)
app.register_action(check_hallucination, name="check_hallucination")
history = [{"role": "user", "content": "Who is the president of Mars?"}]
bot_message = await app.generate_async(messages=history, options={"output_vars": True})
print(bot_message)
# Look for 'check_hallucination' in the log
print(bot_message.output_data)

response=[{'role': 'assistant', 'content': 'As of now, Mars doesn’t have a president or any form of government! Mars is an uninhabited planet in our solar system, currently being explored by various space missions, including rovers sent by NASA and other space agencies. Scientists have studied Mars to understand its geology, climate, and potential for past life, but there are no human settlers or political structures in place there. Discussions about the potential colonization of Mars are ongoing, but any governance on Mars is purely speculative at this point!'}] llm_output=None output_data={'last_user_message': 'Who is the president of Mars?', 'last_bot_message': 'As of now, Mars doesn’t have a president or any form of government! Mars is an uninhabited planet in our solar system, currently being explored by various space missions, including rovers sent by NASA and other space agencies. Scientists have studied Mars to understand its geology, climate, and potential for past life, but t

In [ ]:
config_content = """
models:
  - type: main
    engine: openai
    model: gpt-4o-mini

# Optional: You can specify a separate model for the rail itself 
# if you want to use a faster/cheaper one for the checks.
  - type: self_check_hallucination
    engine: openai
    model: gpt-4o-mini

rails:
  output:
    flows:
      - self check hallucination
      
prompts:
  - task: self_check_hallucination
    content: |
      You are an expert consistency auditor. You are given a "Primary Response" and two "Alternative Responses". 
      Your goal is to determine if the Primary Response contains information that contradicts the Alternative Responses.

      Primary Response: {{ response }}
      Alternative 1: {{ extra_responses[0] }}
      Alternative 2: {{ extra_responses[1] }}

      Is the Primary Response consistent with the alternatives? 
      If it is consistent, answer "no" (meaning no hallucination). 
      If it contradicts the alternatives, answer "yes" (meaning a hallucination was found).
      Answer only with "yes" or "no".
"""

from httpx import options
from nemoguardrails import RailsConfig, LLMRails

config = RailsConfig.from_content(yaml_content=config_content)
rails = LLMRails(config)

# 2. Define the test case
# We use a query that might cause a hallucination or a "trap"
user_query = "What is the current 401k contribution limit for 2026?"

# 3. Execute with high-sampling context
# We pass 'HALLUCINATION_NUM_EXTRA_RESPONSES' to generate more candidates
result = await rails.generate_async(
    messages=[{
        "role": "context", 
        "content": {
                "HALLUCINATION_NUM_EXTRA_RESPONSES": 3, # Direct variable injection
                "temperature": 0.7                      # High temperature for variance
            }
        },
        {"role": "user", "content": user_query}],
    options={"output_vars": True}
)

print(f"User Query: {user_query}")
print(f"Bot Response: {result.response}")

# After your rails.generate_async call:
info = rails.explain()
# 1. Print a summary of all LLM calls (Input, Main, and Hallucination)
info.print_llm_calls_summary()

# 2. Specifically look for the 'self_check_hallucination' task
for call in info.llm_calls:
    if call.task == "self_check_hallucination":
        print(f"--- Hallucination Check Details ---")
        print(f"Prompt used: {call.prompt}")
        print(f"LLM Decision: {call.completion}") # Should be 'yes' or 'no'


User Query: What is the current 401k contribution limit for 2026?
Bot Response: [{'role': 'assistant', 'content': "I'm sorry, but I don't have specific information on 401(k) contribution limits for the year 2026, as my training only goes up until October 2023, and future changes to contribution limits can depend on various factors, including cost-of-living adjustments that may be determined in the future. For the most accurate and up-to-date information, I recommend checking with the IRS or a financial advisor closer to that year. As of 2023, the contribution limit for employee deferrals was $22,500, with a catch-up contribution of $7,500 for those aged 50 and older, but these limits can change."}]
Summary: 1 LLM call(s) took 5.46 seconds and used 217 tokens.

1. Task `general` took 5.46 seconds and used 217 tokens.

